# Part 2: 웹 결과로 근거 추가하기

신입 사원이 방금 Zava의 건강 복리후생이 시장 경쟁력이 있는지 물었습니다. Zava가 내부적으로 무엇을 제공하는지는 알지만, 후반부에 답하려면 최신 업계 벤치마크가 필요합니다. 내부 문서와 웹을 모두 검색할 수 있는 지식 베이스에 딱 맞는 일입니다.

**🎯 미션**
- **Web IQ** 를 사용해 웹을 검색할 수 있는 **MCP 지식 소스** 추가하기
- 3개 소스로 구성된 지식 베이스(HR 문서 + 건강 문서 + 웹) 구축하기
- 하나의 하위 답변은 Zava 내부 건강 문서에서, 다른 하나는 웹페이지에서 나오는 하이브리드 질문하기


## 빌딩 블록

Part 1에서 지식 베이스는 미리 구축된 검색 인덱스만 질의했습니다. 이제 세 번째 소스 유형을 추가합니다. 새로운 Web IQ 서비스에 연결하는 **MCP 서버 지식 소스(MCP Server Knowledge Source)** 입니다.

- **MCP 서버 지식 소스(MCP Server Knowledge Source)**: 모든 원격 MCP 엔드포인트에 연결합니다. 지식 베이스가 질의 시점에 도구처럼 이를 호출합니다.
- **Web IQ**: MCP 서버로 노출되는 Microsoft의 웹 근거 서비스입니다. 순위가 매겨지고 인용이 포함된 웹 결과를 반환합니다.

## 1단계: 환경 변수 설정

Azure 리소스에 대한 구성을 로드합니다. 프로젝트 폴더의 `.env` 파일에 필요한 것의 *거의 대부분* 이 들어 있습니다. Azure AI Search 엔드포인트, Azure OpenAI 자격 증명, 모델 구성입니다.

‼️ `WEB_IQ_KEY` 변수도 필요합니다. 워크샵 진행자가 해당 변수가 들어 있는 파일 링크를 제공합니다. 이를 현재 `.env` 파일 하단에 추가하세요.

```shell
WEB_IQ_KEY="some-value-here"
```

그런 다음 아래 셀을 실행하여 해당 변수를 로드합니다.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
WEB_IQ_KEY = os.environ["WEB_IQ_KEY"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


## 2단계: 세 개의 지식 소스 만들기

이 지식 베이스를 위해 먼저 세 개의 지식 소스를 만듭니다. 첫 번째 노트북에서 만든 것과 동일한 두 개의 인덱스 기반 소스에 더해, Web IQ MCP 서버를 위한 추가 지식 소스 하나입니다.

* `healthdocs-knowledge-source`: `healthdocs` 검색 인덱스를 가리킴
* `hrdocs-knowledge-source`: `hrdocs` 검색 인덱스를 가리킴
* `web-knowledge-source`: 키 기반 인증과 `web` 도구를 사용해 Web IQ 서비스의 원격 MCP 서버를 가리킴

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    McpServerAutoOutputParsing,
    McpServerKnowledgeSource,
    McpServerKnowledgeSourceParameters,
    McpServerStoredHeadersAuthentication,
    McpServerStoredHeadersParameters,
    McpServerTool,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WEB_KNOWLEDGE_SOURCE_NAME = "web-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [HR_KNOWLEDGE_SOURCE_NAME, HEALTH_KNOWLEDGE_SOURCE_NAME, WEB_KNOWLEDGE_SOURCE_NAME]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

web_knowledge_source = McpServerKnowledgeSource(
    name=WEB_KNOWLEDGE_SOURCE_NAME,
    description="Web IQ (live web search)",
    mcp_server_parameters=McpServerKnowledgeSourceParameters(
        server_url="https://api.microsoft.ai/v3/mcp",
        authentication=McpServerStoredHeadersAuthentication(
            stored_headers_parameters=McpServerStoredHeadersParameters(
                {"headers": {"x-apikey": WEB_IQ_KEY}}
            )
        ),
        tools=[McpServerTool(name="web", output_parsing=McpServerAutoOutputParsing())],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=web_knowledge_source)
print("Knowledge sources created")


## 3단계: 멀티 소스 + 웹 지식 베이스 만들기

지식 베이스는 다음을 결합하는 오케스트레이션 계층입니다.

1. 데이터 소스(2단계의 지식 소스들)
2. 질의를 이해하고 답변을 생성하기 위한 LLM(Azure OpenAI)
3. 질의를 처리하고 응답 형식을 지정하는 방법에 대한 구성

이 노트북에서는 연결된 LLM이 원래 사용자 질의에도 답할 수 있도록 "answerSynthesis" `outputMode`를 사용합니다. 또한 "low" `retrievalReasoningEffort`를 사용하는데, 이는 LLM이 질의 계획과 지식 소스 선택에 사용된다는 의미입니다.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-web-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Zava knowledge base combining HR and health documents with access to web search.",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use the HR and health indexes for company policy questions. Use Web IQ web tool for context from public webpages.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


## 4단계: 지식 베이스에 질의

한쪽은 내부 데이터가 필요하고 다른 한쪽은 최신 외부 벤치마크가 필요한 질문을 합니다.

* _"Zava는 어떤 정신 건강 보장을 제공하나요?"_ → `healthdocs`에서 와야 함
* _"기술 기업의 정신 건강 복리후생에 대한 업계 벤치마크는 무엇인가요?"_ → Web IQ를 통해 **웹** 에서 와야 함

지식 베이스는 에이전트형 검색을 사용해 다음을 수행합니다.

1. 질의를 분석하고 두 개의 서로 다른 주제 영역을 인식
2. 집중된 하위 질의로 분해
3. 내부 질문은 검색 인덱스로, 벤치마크 질문은 웹으로 라우팅
4. 세 소스에 걸쳐 검색을 동시 실행
5. 의미 체계 재순위를 사용해 결과 병합

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeSourceParams,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

hrdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    always_query_source=True,
)
healthdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    always_query_source=True,
)
web_knowledge_source_params = KnowledgeSourceParams(
    knowledge_source_name=WEB_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    kind="mcpServer",
)

question = (
    "A new hire wants to know if Zava's benefits are competitive. "
    "What mental health coverage does Zava offer according to our internal health plan docs? "
    "And search the web for current industry benchmarks for mental health benefits at tech companies."
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[hrdocs_knowledge_source_params, healthdocs_knowledge_source_params, web_knowledge_source_params],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
display(Markdown(result.response[0].content[0].text))


### 활동 로그 검토

이 활동 로그에서는 Web IQ에 대한 호출에 해당하는 "mcpServer" 단계와, 각 검색 인덱스로 보낸 질의에 해당하는 "searchIndex" 단계를 볼 수 있습니다.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### 참조 검토

Web IQ 지식 소스의 경우 참조는 `type: "mcpServer"`를 가집니다. 기사 제목, 콘텐츠, URL은 JSON 문자열인 `sourceData.content` 안에 들어 있습니다.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 소스 헌트

위에 출력된 참조를 살펴보세요. 다음을 식별할 수 있나요?

1. 어떤 지식 소스가 **Zava 정신 건강 보장** 질문에 답했나요?
2. 어떤 지식 소스가 **업계 벤치마크** 질문에 답했나요?

## 보너스: Copilot CLI에서 질의하기

모든 지식 베이스는 MCP 서버 엔드포인트를 노출하며, 이는 VS Code나 GitHub Copilot CLI 같은 MCP 호환 클라이언트에 추가할 수 있습니다.
Copilot CLI에서 지식 베이스에 질의해 보고 싶다면 [Copilot CLI 사이드퀘스트](copilot-cli-sidequest.md)의 설정 단계를 거친 후 아래 명령을 실행하여 최신 Zava 지식 베이스를 MCP 서버로 사용하세요.

In [ ]:
print("copilot mcp remove zava-kb")
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
print(f"copilot mcp add zava-kb \"{mcp_url}\" --header \"api-key={AZURE_SEARCH_ADMIN_KEY}\"")
print(f"copilot -i \"{question}\"")

## ✅ 미션 완료

**무엇을 만들었나:**

* ✓ **Web IQ MCP 지식 소스**: MCP 프로토콜을 통해 라이브 웹을 지식 베이스에 연결하는 세 번째 지식 소스로, 지식 베이스가 내부 문서와 함께 실시간 결과를 가져올 수 있게 함.
* ✓ **3개 소스 지식 베이스**: HR 문서, 건강 문서, 라이브 웹에 걸친 단일 지식 베이스로, 각 하위 질문에 적합한 소스를 자동으로 선택하는 에이전트형 라우팅을 갖춤.
* ✓ **하이브리드 검색**: 내부 건강 플랜 문서가 Zava 보장 질문에 답했고, 웹이 라이브 인용과 함께 업계 벤치마크 질문에 답했음.

➡️ [Part 3: Fabric IQ 추가하기](part3-fabric-iq-to-kb.ipynb)로 계속 진행하세요.